In [1]:
import sys

print(sys.executable)

c:\Users\CES\Documents\DATA_ENGINEER\banking-dwh-pipeline\.venv\Scripts\python.exe


In [3]:
import pyspark

print("PySpark:", pyspark.__version__)


PySpark: 4.2.0


In [4]:
from pyspark.sql import SparkSession

spark = SparkSession \
    .builder \
    .appName("BankingDwhPipeline") \
    .config("spark.sql.shuffle.partitions", "4") \
    .master("local[*]") \
    .getOrCreate()

In [5]:
print("Aplicación:", spark.sparkContext.appName)
print("Master:", spark.sparkContext.master)
print(
    "Shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)

Aplicación: BankingDwhPipeline
Master: local[*]
Shuffle partitions: 4


## Lectura del archivo `customers.csv`

En esta sección se crea un DataFrame a partir del archivo CSV de clientes.

Se utilizan las siguientes opciones:

- `header=True`: indica que la primera fila contiene los nombres de las columnas.
- `inferSchema=True`: permite que Spark intente detectar automáticamente los tipos de datos.
- `csv(...)`: indica que la fuente es un archivo CSV.

El DataFrame resultante se almacena en la variable `df_customers`.

In [6]:
df_customers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../data/raw/customers.csv")
)

In [8]:
df_customers.head(5)

[Row(customer_id='CUSXAJI0Y6DPBHS', first_name='Kevin', last_name='Young', email='brauncameron@example.net', city='North Williamville', credit_score=327, created_at=datetime.date(2025, 4, 17)),
 Row(customer_id='CUSHXTHV3A3ZMF8', first_name='Jason', last_name='Clements', email='toddwilliam@example.net', city='Martinezside', credit_score=644, created_at=datetime.date(2020, 2, 23)),
 Row(customer_id='CUSDD4V30T9NT3W', first_name='Randy', last_name='Thompson', email='trevoranderson@example.org', city='Gallowayfurt', credit_score=670, created_at=datetime.date(2025, 6, 22)),
 Row(customer_id='CUSGCX1945NQ4FM', first_name='Laura', last_name='Phillips', email='valdezgeorge@example.com', city='Morrisview', credit_score=573, created_at=datetime.date(2019, 10, 20)),
 Row(customer_id='CUSVG0FN9XUY41I', first_name='Savannah', last_name='Swanson', email='smithluis@example.com', city='Lake Anna', credit_score=332, created_at=datetime.date(2022, 7, 16))]

In [9]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- created_at: date (nullable = true)



In [10]:
df_customers.select("*").show()

+---------------+----------+---------+--------------------+--------------------+------------+----------+
|    customer_id|first_name|last_name|               email|                city|credit_score|created_at|
+---------------+----------+---------+--------------------+--------------------+------------+----------+
|CUSXAJI0Y6DPBHS|     Kevin|    Young|brauncameron@exam...|  North Williamville|         327|2025-04-17|
|CUSHXTHV3A3ZMF8|     Jason| Clements|toddwilliam@examp...|        Martinezside|         644|2020-02-23|
|CUSDD4V30T9NT3W|     Randy| Thompson|trevoranderson@ex...|        Gallowayfurt|         670|2025-06-22|
|CUSGCX1945NQ4FM|     Laura| Phillips|valdezgeorge@exam...|          Morrisview|         573|2019-10-20|
|CUSVG0FN9XUY41I|  Savannah|  Swanson|smithluis@example...|           Lake Anna|         332|2022-07-16|
|CUSOC6UZHR5XFF0|    Ashley|     Wang|reesekendra@examp...|             Amybury|         569|2025-07-22|
|CUSPVN9ER14FFYV|    Morgan|   Miller| narnold@example.

## Inspección del esquema de `customers.csv`

El método `printSchema()` muestra la estructura del DataFrame, incluyendo:

- Nombre de cada columna.
- Tipo de dato detectado por Spark.
- Si la columna permite valores nulos.

El esquema obtenido fue:

```text
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- created_at: date (nullable = true)

In [13]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- created_at: date (nullable = true)




El siguiente paso lógico es conocer **cuántos clientes tiene el archivo**:

```python
total_customers = df_customers.count()

print("Total de clientes:", total_customers)

In [14]:
total_customers = df_customers.count()

print("Total de clientes:", total_customers)

Total de clientes: 50000


Ahora podemos aplicar lo visto en el curso con una primera selección:

```python
df_customers.select(
    "customer_id",
    "first_name",
    "last_name",
    "credit_score"
).show(5, truncate=False)

In [15]:

df_customers.select(
    "customer_id",
    "first_name",
    "last_name",
    "credit_score"
).show(5, truncate=False)

+---------------+----------+---------+------------+
|customer_id    |first_name|last_name|credit_score|
+---------------+----------+---------+------------+
|CUSXAJI0Y6DPBHS|Kevin     |Young    |327         |
|CUSHXTHV3A3ZMF8|Jason     |Clements |644         |
|CUSDD4V30T9NT3W|Randy     |Thompson |670         |
|CUSGCX1945NQ4FM|Laura     |Phillips |573         |
|CUSVG0FN9XUY41I|Savannah  |Swanson  |332         |
+---------------+----------+---------+------------+
only showing top 5 rows
